# **Day 20: Pipelines and ColumnTransformer**
*Module 3 - Data Preparation and Feature Engineering*

A **Pipeline** chains preprocessing and the model into **one estimator**. This is not just tidy code: it is the main defence against leakage, it makes cross-validation and tuning correct, and it lets us deploy one object that turns raw data into predictions.

> A pipeline chains preprocessing and the model into one object, so every step is fitted only on training data. ColumnTransformer applies different preprocessing to different columns (numbers vs categories). Together they prevent leakage.
>
> *Example:* Scaling the spectral bands and one-hot encoding the soil type happen inside the pipeline, so cross-validation repeats them correctly in every fold.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
rng = np.random.default_rng(20)

## **1. A Messy, Realistic Dataset**
House prices with numeric and categorical columns, missing values and skew.

In [2]:
n = 2000
df = pd.DataFrame({
    "area_m2": rng.lognormal(4.6, 0.4, n).round(1),
    "rooms": rng.integers(1, 7, n).astype(float),
    "age_years": rng.uniform(0, 60, n).round(),
    "dist_center_km": rng.gamma(2, 3, n).round(2),
    "city": rng.choice(["Kolkata", "Dhanbad", "Ranchi", "Patna"], n, p=[0.4, 0.2, 0.2, 0.2]),
    "type": rng.choice(["flat", "house", "villa"], n, p=[0.6, 0.3, 0.1]),
    "condition": rng.choice(["poor", "fair", "good", "excellent"], n),
})
city_eff = df.city.map({"Kolkata": 0.5, "Dhanbad": 0.0, "Ranchi": 0.1, "Patna": 0.05})
type_eff = df.type.map({"flat": 0.0, "house": 0.15, "villa": 0.5})
cond_eff = df.condition.map({"poor": -0.2, "fair": 0.0, "good": 0.1, "excellent": 0.25})
df["price_lakh"] = np.exp(1.0 + 0.9 * np.log(df.area_m2) - 0.006 * df.age_years - 0.03 * df.dist_center_km
                          + city_eff + type_eff + cond_eff + rng.normal(0, 0.15, n)).round(2)
for c in ["area_m2", "age_years", "type"]:
    df.loc[rng.random(n) < 0.08, c] = np.nan
X, y = df.drop(columns="price_lakh"), df.price_lakh
X.head()

,area_m2,rooms,age_years,dist_center_km,city,type,condition
0,86.2,1.0,44.0,1.89,Dhanbad,house,poor
1,161.0,5.0,4.0,3.09,Kolkata,house,poor
2,NaN,3.0,25.0,7.53,Kolkata,villa,excellent
3,112.9,5.0,38.0,2.49,Patna,flat,good
4,117.4,2.0,17.0,4.06,Kolkata,flat,good


## **2. ColumnTransformer + Pipeline**

In [3]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge

num_cols = ["area_m2", "rooms", "age_years", "dist_center_km"]
nom_cols = ["city", "type"]
ord_cols = ["condition"]

numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
nominal = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
ordinal = OrdinalEncoder(categories=[["poor", "fair", "good", "excellent"]])

preprocess = ColumnTransformer([("num", numeric, num_cols), ("nom", nominal, nom_cols), ("ord", ordinal, ord_cols)])
model = Pipeline([("prep", preprocess), ("reg", Ridge(alpha=1.0))])
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the outp

In Jupyter the pipeline is displayed as an interactive diagram. Now everything is one estimator:

In [4]:
from sklearn.model_selection import train_test_split, cross_val_score
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
model.fit(X_tr, y_tr)
print(f"Test R2: {model.score(X_te, y_te):.3f}")
print("CV R2  :", cross_val_score(model, X_tr, y_tr, cv=5).mean().round(3), "(imputer/scaler refit inside every fold)")
print("Features after preprocessing:", list(model[:-1].get_feature_names_out()))

Test R2: 0.775
CV R2  : 0.8 (imputer/scaler refit inside every fold)
Features after preprocessing: ['num__area_m2', 'num__rooms', 'num__age_years', 'num__dist_center_km', 'nom__city_Dhanbad', 'nom__city_Kolkata', 'nom__city_Patna', 'nom__city_Ranchi', 'nom__type_flat', 'nom__type_house', 'nom__type_villa', 'ord__condition']


## **3. Predicting the Log of a Skewed Target**
Prices are right-skewed and effects are multiplicative. `TransformedTargetRegressor` fits on $\log(y)$ and automatically converts predictions back.

In [5]:
from sklearn.compose import TransformedTargetRegressor
log_model = TransformedTargetRegressor(regressor=model, func=np.log, inverse_func=np.exp)
print("Ridge on y      CV R2:", cross_val_score(model, X_tr, y_tr, cv=5).mean().round(3))
print("Ridge on log(y) CV R2:", cross_val_score(log_model, X_tr, y_tr, cv=5).mean().round(3))

Ridge on y      CV R2: 0.8


Ridge on log(y) CV R2: 0.801


## **4. Custom Transformers**
`FunctionTransformer` wraps a stateless function. For transformers that **learn** something in `fit`, write a class.

In [6]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin

log_tf = FunctionTransformer(np.log1p, feature_names_out="one-to-one")

class RatioFeatures(TransformerMixin, BaseEstimator):
    """Adds area per room. Stateless, but shows the pattern."""
    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]; return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X["area_per_room"] = X["area_m2"] / X["rooms"].clip(lower=1)
        return X
    def get_feature_names_out(self, input_features=None):
        return np.r_[np.asarray(input_features), ["area_per_room"]]

class Winsorizer(TransformerMixin, BaseEstimator):
    """Learns percentiles on TRAIN data and clips with them (stateful)."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper
    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)   # remember names for get_feature_names_out
        X = np.asarray(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.lo_, self.hi_ = np.nanquantile(X, self.lower, axis=0), np.nanquantile(X, self.upper, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)
    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return getattr(self, "feature_names_in_", np.array([f"x{i}" for i in range(self.n_features_in_)], dtype=object))

numeric2 = Pipeline([("impute", SimpleImputer(strategy="median")), ("clip", Winsorizer()),
                     ("log", log_tf), ("scale", StandardScaler())])
prep2 = ColumnTransformer([("num", numeric2, num_cols), ("nom", nominal, nom_cols), ("ord", ordinal, ord_cols)])
model2 = TransformedTargetRegressor(Pipeline([("prep", prep2), ("reg", Ridge())]), func=np.log, inverse_func=np.exp)
print("Custom pipeline CV R2:", cross_val_score(model2, X_tr, y_tr, cv=5).mean().round(3))

Custom pipeline CV R2: 0.864


## **5. Pandas Output for Inspection**

In [7]:
prep_df = prep2.set_output(transform="pandas")
prep_df.fit_transform(X_tr).head()

,num__area_m2,num__rooms,num__age_years,num__dist_center_km,nom__city_Dhanbad,nom__city_Kolkata,nom__city_Patna,nom__city_Ranchi,nom__type_flat,nom__type_house,nom__type_villa,ord__condition
582,0.942001,0.885478,-0.095296,-0.048351,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
159,0.414643,-0.054307,0.410584,0.787921,0.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0
1827,0.917726,0.885478,0.291029,0.762300,0.0,0.0,0.0,1.0,1.0,0.0,0.0,2.0
318,1.396487,-0.721096,0.248591,-0.443502,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
708,-1.948389,-0.721096,0.588156,-0.500486,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


## **6. Tuning Preprocessing and Model Together**
Parameter names use `step__substep__param`.

In [8]:
from sklearn.model_selection import GridSearchCV
grid = {"regressor__prep__num__impute__strategy": ["mean", "median"],
        "regressor__prep__num__clip__upper": [0.95, 0.99, 1.0],
        "regressor__reg__alpha": [0.1, 1, 10]}
gs = GridSearchCV(model2, grid, cv=5, scoring="r2", n_jobs=-1).fit(X_tr, y_tr)
print("Best params:", gs.best_params_)
print(f"Best CV R2: {gs.best_score_:.4f} | Test R2: {gs.score(X_te, y_te):.4f}")

Best params: {'regressor__prep__num__clip__upper': 1.0, 'regressor__prep__num__impute__strategy': 'mean', 'regressor__reg__alpha': 0.1}
Best CV R2: 0.8663 | Test R2: 0.8275


## **7. Save and Load the Whole Pipeline**
One file contains imputation statistics, encoders, scalers and the model. New raw data in, predictions out.

In [9]:
import joblib
joblib.dump(gs.best_estimator_, "house_price_pipeline.joblib")
loaded = joblib.load("house_price_pipeline.joblib")
new_houses = pd.DataFrame({"area_m2": [85, np.nan], "rooms": [3, 2], "age_years": [5, 40], "dist_center_km": [2.5, 12],
                           "city": ["Kolkata", "Siliguri"], "type": ["flat", None], "condition": ["good", "fair"]})
print("Predicted prices (lakh):", loaded.predict(new_houses).round(1), "<- handles NaN and an unseen city")

Predicted prices (lakh): [245. 117.] <- handles NaN and an unseen city


# **Part 2: Going Deeper**

### A pipeline that chooses the model
The final step itself can be a hyperparameter.

In [10]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
pipe = Pipeline([("prep", prep2), ("reg", Ridge())])
search = GridSearchCV(TransformedTargetRegressor(pipe, func=np.log, inverse_func=np.exp),
                      [{"regressor__reg": [Ridge()], "regressor__reg__alpha": [0.1, 1, 10]},
                       {"regressor__reg": [KNeighborsRegressor()], "regressor__reg__n_neighbors": [5, 15]},
                       {"regressor__reg": [HistGradientBoostingRegressor(random_state=0)], "regressor__reg__max_iter": [100, 300]}],
                      cv=5, n_jobs=-1).fit(X_tr, y_tr)
res = pd.DataFrame(search.cv_results_)[["param_regressor__reg", "mean_test_score"]].sort_values("mean_test_score", ascending=False)
res["param_regressor__reg"] = res.param_regressor__reg.astype(str).str[:40]
print(res.head(6).to_string(index=False))

                    param_regressor__reg  mean_test_score
                                 Ridge()         0.863627
                                 Ridge()         0.863510
                                 Ridge()         0.861781
HistGradientBoostingRegressor(random_sta         0.835385
HistGradientBoostingRegressor(random_sta         0.824371
                   KNeighborsRegressor()         0.708706


### Caching expensive steps
`Pipeline(..., memory="cache_dir")` stores fitted transformers so a grid search over the **model** parameters does not refit the same preprocessing again and again.

### Pipelines and leakage, one more time
Anything that **learns from data** (imputers, scalers, encoders, feature selection, resampling such as SMOTE on Day 21, PCA) belongs **inside** the pipeline. Only stateless, row-wise operations (unit conversion, parsing dates) can safely be done before splitting.

## **Practice Problems**
1. Replace `OneHotEncoder` by `TargetEncoder` for `city` and compare CV R2.
2. Use `make_column_selector(dtype_include="number")` instead of listing numeric columns by hand.
3. Add a `RatioFeatures` step before the `ColumnTransformer` (and add `area_per_room` to `num_cols`). Does it help?
4. *(Advanced)* Replace Ridge by `HistGradientBoostingRegressor` and remove the scaler and imputer (HGB handles NaN). Use `OrdinalEncoder` for categoricals with `categorical_features="from_dtype"` or by index. Compare.

In [11]:
# Solution 1
from sklearn.preprocessing import TargetEncoder
prep_te = ColumnTransformer([("num", numeric2, num_cols), ("city", TargetEncoder(), ["city"]),
                             ("type", nominal, ["type"]), ("ord", ordinal, ord_cols)])
m_te = TransformedTargetRegressor(Pipeline([("prep", prep_te), ("reg", Ridge())]), func=np.log, inverse_func=np.exp)
print("TargetEncoder for city CV R2:", cross_val_score(m_te, X_tr, y_tr, cv=5).mean().round(3))
# Solution 2
prep_sel = ColumnTransformer([("num", numeric2, make_column_selector(dtype_include="number")),
                              ("nom", nominal, nom_cols), ("ord", ordinal, ord_cols)])
print("Selector-based CV R2:", cross_val_score(TransformedTargetRegressor(Pipeline([("prep", prep_sel), ("reg", Ridge())]), func=np.log, inverse_func=np.exp), X_tr, y_tr, cv=5).mean().round(3))
# Solution 3
prep_r = ColumnTransformer([("num", numeric2, num_cols + ["area_per_room"]), ("nom", nominal, nom_cols), ("ord", ordinal, ord_cols)])
m_r = TransformedTargetRegressor(Pipeline([("ratio", RatioFeatures()), ("prep", prep_r), ("reg", Ridge())]), func=np.log, inverse_func=np.exp)
print("With area_per_room CV R2:", cross_val_score(m_r, X_tr, y_tr, cv=5).mean().round(3))
# Solution 4
cat_cols = nom_cols + ord_cols
prep_h = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, encoded_missing_value=np.nan), cat_cols)],
                           remainder="passthrough")
hgb = Pipeline([("prep", prep_h), ("reg", HistGradientBoostingRegressor(categorical_features=[0, 1, 2], random_state=0))])
print("HGB native categorical + NaN CV R2:", cross_val_score(TransformedTargetRegressor(hgb, func=np.log, inverse_func=np.exp), X_tr, y_tr, cv=5).mean().round(3))

TargetEncoder for city CV R2: 0.862


Selector-based CV R2: 0.864


With area_per_room CV R2: 0.863


HGB native categorical + NaN CV R2: 0.836


## **Key References**
- Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *JMLR*, 12, 2825-2830.
- Buitinck, L. et al. (2013). API design for machine learning software: experiences from the scikit-learn project. arXiv:1309.0238.
- Sculley, D. et al. (2015). Hidden technical debt in machine learning systems. *NeurIPS 2015*.